# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [1]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Run the cell below every time to activate the installed environment. 

In [2]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

/bin/bash: line 1: ./.venv/bin/activate: No such file or directory


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [3]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768
SEED        = 13
EVAL_N_MCQ  = 50
EVAL_N_FREE = 50
# Local-eval tolerance estimate (DEFAULT — overridden per-question by
# infer_precision_from_question when the question text specifies precision).
# 1e-2 (1% relative) measures "the model basically got it" — generous because
# the model tends to round numerical answers to 2 decimal places. Set to 1e-8
# to recover the strict judger score, or to 1e-3 for a moderate middle ground.
EVAL_PRECISION = 1e-2
SUBSET_PATH = "results/eval_subset.json"

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [4]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4.1 Preprocessing

Before sending a question to the model, we apply deterministic cleanup:

1. **LaTeX repair** — `repair_latex(text)` re-attaches missing backslashes to
   known math commands. The dataset has systematic LaTeX corruption (e.g.
   `int_{-infty}^{+infty} frac{a}{s^2+a^2}` should be
   `\int_{-\infty}^{+\infty} \frac{a}{s^2+a^2}`). Even strong external
   reasoners cannot recover from this on free-form questions; on MCQ they
   can back-infer from options at the cost of accuracy.
2. **Question preprocessing** — `preprocess_question(item)` wraps
   `repair_latex` and applies it to the question text and (if present) each
   option string. Returns a new dict; does not mutate the input.

Both functions are pure (no model dependency) and idempotent.


In [5]:
# LaTeX repair + Unicode-math normalization (v2.9)

# v2.9: Unicode math symbols → LaTeX. Some questions in both public and private
# sets contain raw Unicode codepoints (∫ ∑ √ π ≥ ≤ etc.) instead of LaTeX
# commands. Convert FIRST, before repair_latex.
_UNICODE_MATH_MAP = {
    # Calculus / analysis
    "∫": r"\int", "∬": r"\iint", "∭": r"\iiint", "∮": r"\oint",
    "∑": r"\sum", "∏": r"\prod",
    "√": r"\sqrt", "∞": r"\infty",
    "∂": r"\partial", "∇": r"\nabla",
    # Greek lowercase
    "α": r"\alpha", "β": r"\beta", "γ": r"\gamma", "δ": r"\delta",
    "ε": r"\epsilon", "ζ": r"\zeta", "η": r"\eta", "θ": r"\theta",
    "ι": r"\iota", "κ": r"\kappa", "λ": r"\lambda", "μ": r"\mu",
    "ν": r"\nu", "ξ": r"\xi", "π": r"\pi", "ρ": r"\rho",
    "σ": r"\sigma", "τ": r"\tau", "υ": r"\upsilon", "φ": r"\phi",
    "χ": r"\chi", "ψ": r"\psi", "ω": r"\omega",
    # Greek uppercase (skip ones identical to Latin: A B E Z H I K M N O P T X Y)
    "Γ": r"\Gamma", "Δ": r"\Delta", "Θ": r"\Theta", "Λ": r"\Lambda",
    "Ξ": r"\Xi", "Π": r"\Pi", "Σ": r"\Sigma", "Υ": r"\Upsilon",
    "Φ": r"\Phi", "Ψ": r"\Psi", "Ω": r"\Omega",
    # Relations
    "≥": r"\ge", "≤": r"\le", "≠": r"\ne", "≈": r"\approx",
    "≡": r"\equiv", "±": r"\pm", "∓": r"\mp",
    "⋅": r"\cdot", "×": r"\times", "÷": r"\div",
    # Sets
    "∈": r"\in", "∉": r"\notin", "⊂": r"\subset", "⊆": r"\subseteq",
    "⊃": r"\supset", "⊇": r"\supseteq", "∅": r"\emptyset",
    "∪": r"\cup", "∩": r"\cap",
    # Logic
    "∀": r"\forall", "∃": r"\exists", "¬": r"\neg",
    "∧": r"\land", "∨": r"\lor",
    # Arrows
    "→": r"\to", "←": r"\leftarrow", "↔": r"\leftrightarrow",
    "⇒": r"\Rightarrow", "⇐": r"\Leftarrow", "⇔": r"\Leftrightarrow",
    "↦": r"\mapsto",
    # Number sets
    "ℝ": r"\mathbb{R}", "ℤ": r"\mathbb{Z}", "ℚ": r"\mathbb{Q}",
    "ℕ": r"\mathbb{N}", "ℂ": r"\mathbb{C}",
    # Degree
    "°": r"^{\circ}",
}


def normalize_unicode_math(text: str) -> str:
    """Convert Unicode math symbols to LaTeX commands.

    For commands ending in a letter (e.g. \\int, \\alpha), insert a space when
    immediately followed by another alpha character so \\intx doesn't form.

    Uses a lambda for the regex substitution because re.sub treats the
    replacement string as a template where backslash is special (so plain
    re.sub(..., r'\\int ', ...) raises 'bad escape \\i').
    """
    if not text:
        return text
    for sym, latex in _UNICODE_MATH_MAP.items():
        if sym not in text:
            continue
        if latex.startswith("\\") and latex[-1].isalpha():
            replacement = latex + " "
            text = re.sub(re.escape(sym) + r"(?=[A-Za-z])", lambda _: replacement, text)
        text = text.replace(sym, latex)
    return text


# Common LaTeX commands appearing in the dataset's mathematical formulas.
LATEX_CMDS = [
    # Calculus / analysis
    "int", "iint", "iiint", "oint", "sum", "prod", "lim",
    "infty", "partial",
    # Fractions / roots
    "frac", "dfrac", "tfrac", "sqrt", "binom",
    # Common functions
    "sin", "cos", "tan", "cot", "sec", "csc",
    "arcsin", "arccos", "arctan",
    "sinh", "cosh", "tanh",
    "log", "ln", "exp",
    # Greek (lowercase)
    "alpha", "beta", "gamma", "delta", "epsilon", "zeta", "eta",
    "theta", "iota", "kappa", "lambda", "mu", "nu",
    "xi", "pi", "rho", "sigma", "tau", "phi", "chi", "psi", "omega",
    # Greek (uppercase)
    "Gamma", "Delta", "Theta", "Lambda", "Pi", "Sigma", "Phi", "Psi", "Omega",
    # Operators / relations
    "pm", "mp", "times", "cdot", "leq", "geq", "neq",
    "approx", "equiv", "sim", "to", "in", "notin",
    "subset", "supset", "cup", "cap",
    # Formatting
    "mathbb", "mathrm", "mathbf", "mathcal",
    "left", "right", "text",
]

# Strict context-aware pattern: only repair when the command is followed by a
# math-syntax character. This prevents false positives on standalone English
# words ("the tan of x", "in this case", "to compute") while still catching
# every real corruption in the dataset, where bare commands are always
# followed by `{`, `_`, `^`, `(`, `)`, or `}`.
_LATEX_MATH_CONTEXT = r"[{}_^()]"
_LATEX_PATTERNS = [
    (re.compile(rf"(?<!\\)\b{cmd}(?={_LATEX_MATH_CONTEXT})"), rf"\\{cmd}")
    for cmd in LATEX_CMDS
]


def repair_latex(text: str) -> str:
    """Re-attach missing backslashes to known LaTeX commands in math context.

    Idempotent: repair_latex(repair_latex(x)) == repair_latex(x).
    """
    for pattern, repl in _LATEX_PATTERNS:
        text = pattern.sub(repl, text)
    return text


def preprocess_question(item: dict) -> dict:
    """v2.9 chain: normalize_unicode_math → repair_latex on question + options."""
    out = dict(item)
    q = normalize_unicode_math(item["question"])
    out["question"] = repair_latex(q)
    if item.get("options"):
        out["options"] = [repair_latex(normalize_unicode_math(opt)) for opt in item["options"]]
    return out


In [6]:
# Verify on real dataset samples
print("LaTeX repair on Q1 (MCQ with mangled formula)")
q1 = data[1]
print(f"  before: {q1['question']}")
print(f"  after : {repair_latex(q1['question'])}")
print(f"  options[0:3] before: {q1['options'][:3]}")
print(f"  options[0:3] after : {[repair_latex(o) for o in q1['options'][:3]]}")

print("\nFalse-positive check (English words that collide with command names)")
for s in ["the tan of x", "in this case", "to compute", "consider tangent", "Bob is tan"]:
    out = repair_latex(s)
    flag = " <= CHANGED" if out != s else ""
    print(f"  {s!r:35} -> {out!r}{flag}")

print("\nIdempotence check")
sample = "int_{-infty}^{+infty} frac{a}{s^2+a^2}"
once  = repair_latex(sample)
twice = repair_latex(once)
print(f"  repair_latex once == twice: {once == twice}")
print(f"  result: {once}")

LaTeX repair on Q1 (MCQ with mangled formula)
  before: $int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $
  after : $\int_{-\infty}^{+\infty} \frac{a^{3/2}}{s^2+a^2} ds = $
  options[0:3] before: ['$0$', '$frac{1}{a}$', '$frac{3}{a}$']
  options[0:3] after : ['$0$', '$\\frac{1}{a}$', '$\\frac{3}{a}$']

False-positive check (English words that collide with command names)
  'the tan of x'                      -> 'the tan of x'
  'in this case'                      -> 'in this case'
  'to compute'                        -> 'to compute'
  'consider tangent'                  -> 'consider tangent'
  'Bob is tan'                        -> 'Bob is tan'

Idempotence check
  repair_latex once == twice: True
  result: \int_{-\infty}^{+\infty} \frac{a}{s^2+a^2}


## 4.2 Prompt Construction

We define the system prompts and the prompt-building functions:

- `SYSTEM_PROMPT_MATH` — solve-and-box instructions for free-form questions.
- `SYSTEM_PROMPT_MCQ` — letter-only selection for multiple-choice questions.
- `MCQ_STAGE2_INSTRUCTION` — short reconciliation instruction for the two-stage
  MCQ flow (used only when stage-1's answer doesn't match any option).
- `build_prompt(question, options)` — **baseline** single-call prompt
  construction. Returns `(system, user)` where `user` includes the options
  inline for MCQ. Kept unchanged for direct comparison against v2.
- `select_prompt(item)` — **v2** conditional prompt routing. For MCQ, returns
  the *stage-1* prompts (question only, no options shown) so the model must
  derive its own answer before being anchored by the option set. For
  free-form, routes by question complexity (multi-part / long / short).
- `verify_against_options(response, options)` — uses `Judger.auto_judge` to
  check whether the stage-1 answer matches any option. Returns the matching
  letter or `None`.
- `build_mcq_stage2_messages(item, stage1_response)` — builds the multi-turn
  chat for the stage-2 reconciliation call, invoked only when
  `verify_against_options` returned `None`.

`build_prompt` is the **baseline pipeline**; `select_prompt` plus the
two-stage MCQ helpers are the **v2 pipeline**. Both coexist so we can A/B
test cleanly.


In [7]:
# ============================================================================
# System prompts — v2.9, post-research-verification (Agent 2 findings Q1-Q9)
# ============================================================================

# Generic short free-form (also reused for MCQ-hidden-options stage-1 and as
# the system role in build_mcq_stage2_messages). v2.9: shortened to canonical
# Qwen3-Thinking line per Q2 — multipart hint moved to SYSTEM_PROMPT_MULTIPART
# and to user-prompt suffix where it belongs.
SYSTEM_PROMPT_MATH = (
    "Please reason step by step, and put your final answer within \\boxed{}."
)

# MCQ visible-options. v2.9 Q9: format constraint MOVED OUT of system prompt to
# the user-prompt tail (see _append_mcq_letter_instruction). System is now the
# canonical minimal line so format-attention isn't diluted across <think>.
# v2.9 Q6: charitable-interpretation clause removed — repair_latex already
# handles the 5-7% bare-word LaTeX, so the unconditional prompt-side fallback
# was paying tokens on every MCQ for marginal coverage. Kept (shortened) only
# in MCQ_STAGE2_INSTRUCTION where it fires after a failure (targeted use).
SYSTEM_PROMPT_MCQ = SYSTEM_PROMPT_MATH

# MCQ stage-2 reconciliation user message. Fired only when stage-1
# verify_against_options_lenient returned None. v2.9 Q6: shortened charity
# clause. Format instruction at the tail per Q9.
MCQ_STAGE2_INSTRUCTION = (
    "Your previous answer does not match any of the options shown above. The "
    "question text may have minor transcription errors; pick the option that "
    "makes the problem mathematically sensible. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. "
    "\\boxed{C}. Do not include the option text or any other content inside "
    "the box."
)

# Long free-form (qlen > 500), CoRe-style: restate + list conditions before
# solving. Used when question is long AND not multipart AND not olympiad.
SYSTEM_PROMPT_LONG = (
    "Before solving, restate the problem in your own words and list every given "
    "condition and the quantity to find. Then solve.\n\n"
    "Please reason step by step, and put your final answer within \\boxed{}."
)

# Multipart free-form. v2.9 Q1: switched from "one big comma-separated \boxed{}"
# to "N separate \boxed{} blocks" — that's the judger's native extraction path
# (extract_boxed_answer concatenates the last contiguous group of \boxed{...}
# answers) and avoids comma-collision with tuple/set answers like (1,2).
SYSTEM_PROMPT_MULTIPART = (
    "This problem contains multiple sub-answers marked by [ANS] placeholders "
    "in the question. Identify each sub-question, then solve them in the order "
    "they appear.\n\n"
    "After completing all reasoning, output your answers as separate "
    "\\boxed{...} blocks in left-to-right order, one per [ANS] marker. "
    "For example, for a 3-part question, end your response with: "
    "\\boxed{ans1} \\boxed{ans2} \\boxed{ans3}.\n\n"
    "If any sub-question shows inline lettered options (A. ... B. ... C. ...), "
    "output ONLY the letter of your chosen option inside that sub-question's "
    "\\boxed{}, e.g. \\boxed{C}. Do not include the option text or any other "
    "content inside that box."
)


# ============================================================================
# v2.9 NEW: embedded-MCQ detection (free-form record with inline lettered options)
# ============================================================================
# Pattern: [ANS] within 40 chars of whitespace/punct → "A. <content>" then
# "B. <content>". MUST be gated by n_ans <= 1 in select_prompt — multipart
# questions often use A./B./C. as subpart labels which look like MCQ options
# structurally and would false-positive without the gate.
_EMBEDDED_MCQ_RE = re.compile(
    r"\[ANS\][\s\.\$\"',]{0,40}"
    r"A[\.\)][\s]+[^\n]{3,300}\s+"
    r"B[\.\)][\s]+[^\n]{3,300}",
    re.DOTALL,
)

# Capture inline lettered options "A. ... B. ... C. ..." after the [ANS] anchor.
_INLINE_OPTS_RE = re.compile(
    r"([A-J])[\.\)]\s+(.+?)(?=\s+[A-J][\.\)]\s+|\s*$)",
    re.DOTALL,
)


def detect_embedded_mcq(question: str) -> Optional[list]:
    """Detect free-form question with embedded inline MCQ options.

    Returns extracted list of option text strings (in A/B/C/... order) if
    detected; None otherwise. Caller is responsible for the n_ans <= 1 gate.
    """
    if not _EMBEDDED_MCQ_RE.search(question):
        return None
    m_after = re.search(r"\[ANS\]([\s\S]+)$", question)
    if not m_after:
        return None
    after_ans = m_after.group(1)
    opt_matches = _INLINE_OPTS_RE.findall(after_ans)
    if len(opt_matches) < 2:
        return None
    letters = [m[0] for m in opt_matches]
    expected = [chr(65 + i) for i in range(len(letters))]
    if letters != expected:
        return None
    return [text.strip() for _, text in opt_matches]


# ============================================================================
# v2.9 NEW: format-constraint detection (drives user-prompt tail echo)
# ============================================================================
# Wordlist is intentionally CONSERVATIVE (adversarial review: "simplify" alone
# is ~25% false-positive). Only match more-specific "in simplest form" / etc.
_FORMAT_CONSTRAINT_PATTERNS = [
    (re.compile(r"\b(?:round(?:ed)? to|correct to|to)\s+(\w+)\s+decimal\s+places?\b", re.I),
        "round to {1} decimal places"),
    (re.compile(r"\b(?:round(?:ed)? to|correct to|with|to)\s+(\w+)\s+(?:significant\s+(?:figures?|digits?)|sig\.?\s*figs?)\b", re.I),
        "give {1} significant figures"),
    (re.compile(r"\bnearest\s+(tenth|hundredth|thousandth|ten-thousandth|hundred-thousandth)\b", re.I),
        "round to the nearest {1}"),
    (re.compile(r"\bnearest\s+(?:integer|whole(?:\s+number)?)\b", re.I),
        "round to the nearest integer"),
    # v2.9-polish #2: nearest cent / nearest dollar (financial problems, ~3 records)
    (re.compile(r"\bnearest\s+(cent|dollar)\b", re.I),
        "round to the nearest {1}"),
    (re.compile(r"\bexact\s+(?:value|answer|form)\b", re.I),
        "give the exact value (do not round)"),
    # v2.9-polish #4: "do not round" with negative lookahead — skip when it refers
    # to intermediate steps (the question is asking the OUTPUT to be rounded but
    # intermediate work not rounded; the "exact value" intent doesn't apply to output)
    (re.compile(r"\bdo\s+not\s+round(?!\s+(?:in\s+)?intermediate|\s+(?:in\s+)?(?:any\s+)?calculations?|\s+(?:any\s+)?intermediate)", re.I),
        "do not round"),
    (re.compile(r"\bno\s+rounding\b", re.I),
        "do not round"),
    # v2.9-polish #2: do not simplify (1 record)
    (re.compile(r"\bdo\s+not\s+simplify\b", re.I),
        "do not simplify"),
    # v2.9-polish #3: "in degrees" — exclude descriptive contexts
    # (Celsius/Fahrenheit/Rankine are temperature SCALE adjectives, not output-format requests).
    # Use negative lookahead.
    (re.compile(r"\bin\s+degrees\b(?!\s*(?:Celsius|Fahrenheit|Rankine|C\b|F\b))", re.I),
        "express the answer in degrees"),
    (re.compile(r"\bin\s+radians\b", re.I),
        "express the answer in radians"),
    (re.compile(r"\bas\s+a\s+fraction\b", re.I),
        "express the answer as a fraction"),
    (re.compile(r"\bas\s+a\s+decimal\b", re.I),
        "express the answer as a decimal"),
    (re.compile(r"\bin\s+(?:lowest|simplest)\s+(?:terms|form)\b", re.I),
        "express the answer in simplest form"),
    (re.compile(r"\bsimplified\s+form\b", re.I),
        "express the answer in simplified form"),
    (re.compile(r"\bin\s+scientific\s+notation\b", re.I),
        "express the answer in scientific notation"),
    (re.compile(r"\bin\s+interval\s+notation\b", re.I),
        "express the answer in interval notation"),
    (re.compile(r"\bas\s+a\s+percent(?:age)?\b", re.I),
        "express the answer as a percent"),
    # v2.9-polish #2: structural form requests (vertex/factored/standard)
    (re.compile(r"\bin\s+(vertex|factored|standard)\s+form\b", re.I),
        "express the answer in {1} form"),
]


def detect_format_constraints(question: str) -> list:
    """Extract format-constraint hints from a question; dedup-preserving order."""
    seen = []
    for pat, template in _FORMAT_CONSTRAINT_PATTERNS:
        m = pat.search(question)
        if not m:
            continue
        hint = template.format(*([None] + list(m.groups()))) if "{1}" in template else template
        if hint not in seen:
            seen.append(hint)
    return seen


def _append_format_constraints(user_prompt: str, question_text: str) -> str:
    """Append a 'Required output format: ...' suffix if any constraints found.

    Q5: end-of-user-prompt placement is research-backed (Spotlight Your
    Instructions, arXiv 2505.12025; PASTA, arXiv 2311.02262). Label
    'Required output format' (vs 'Constraints') primes the model toward the
    emission path rather than the solving path.
    """
    constraints = detect_format_constraints(question_text)
    if not constraints:
        return user_prompt
    return user_prompt + "\n\nRequired output format: " + "; ".join(constraints) + "."


def _append_multipart_slotcount(user_prompt: str, n_ans: int) -> str:
    """Append a slot-count instruction for multipart questions.

    v2.9 Q1: N separate \\boxed{} blocks (judger-native), not single
    comma-separated \\boxed{}.
    """
    return user_prompt + (
        f"\n\nThis problem has {n_ans} parts. Provide exactly {n_ans} "
        f"\\boxed{{...}} blocks in left-to-right order, one per [ANS] marker."
    )


def _append_mcq_letter_instruction(user_prompt: str) -> str:
    """v2.9 Q9: MCQ answer-format constraint at user-prompt tail.

    Moved from SYSTEM_PROMPT_MCQ to here so format attention survives past
    the <think> block. Applies to both visible-MCQ and embedded-MCQ branches.
    """
    return user_prompt + (
        "\n\nOutput ONLY the letter of your chosen option inside \\boxed{}, "
        "e.g. \\boxed{C}. Do not include the option text or any other "
        "content inside the box."
    )


# ============================================================================
# Heuristic: does this MCQ require options to be visible in the prompt?
# (UNCHANGED from v2.8 — adversarial review confirmed proposed widening yields
# 0 net new hits in both public and private datasets.)
# ============================================================================
_OPTIONS_REFERENCED = re.compile(
    r"\b(?:"
    r"which (?:of the )?(?:following|options?|statements?|choices?)"
    r"|(?:all|none) of the above"
    r"|select all"
    r"|what (?:is|are) (?:correct|right|true)"
    r"|determine the (?:corresponding )?output"
    r")\b",
    re.IGNORECASE,
)
_OPT_SELF_REFERENCE = re.compile(r"\b(?:all|none) of the above\b", re.IGNORECASE)


def mcq_needs_options(question: str, options: list) -> bool:
    """True iff the question text or option set requires options to be visible."""
    if _OPTIONS_REFERENCED.search(question):
        return True
    return any(_OPT_SELF_REFERENCE.search(opt) for opt in options)


def build_prompt(question: str, options: Optional[list]) -> tuple:
    """Baseline single-call prompt construction (kept for direct comparison)."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# ============================================================================
# v2.9 select_prompt: routing order (CRITICAL — multipart FIRST per audit)
#   1. Free-form, multipart (n_ans > 1)        → MULTIPART
#   2. Free-form, embedded MCQ (n_ans <= 1)    → MCQ visible (single-stage)
#   3. Free-form, olympiad (no [ANS])          → no system prompt (MathArena style)
#   4. Free-form, long (qlen > 500)            → LONG
#   5. Free-form, short                        → MATH
#   6. MCQ flagged by mcq_needs_options        → MCQ visible (single-stage)
#   7. MCQ otherwise                           → MATH, hidden options (two-stage)
#
# Returns (system, user, options_visible). system may be None for olympiad.
# ============================================================================
def select_prompt(item: dict) -> tuple:
    question = item["question"]
    options  = item.get("options")

    if not options:
        n_ans = question.count("[ANS]")
        qlen  = len(question)

        # 1. Multipart FIRST — multipart often has A./B./C. subpart labels that
        #    would false-positive the embedded-MCQ detector.
        if n_ans > 1:
            user = _append_format_constraints(
                _append_multipart_slotcount(question, n_ans),
                question,
            )
            return SYSTEM_PROMPT_MULTIPART, user, False

        # 2. Embedded MCQ (free-form record with inline A./B./... options).
        #    Only fires when n_ans <= 1, gated above. Question already contains
        #    the options inline; append MCQ letter instruction + format constraints.
        embedded_options = detect_embedded_mcq(question)
        if embedded_options is not None:
            user = _append_format_constraints(
                _append_mcq_letter_instruction(question),
                question,
            )
            return SYSTEM_PROMPT_MCQ, user, True

        # 3. Olympiad branch — no [ANS] marker AND qlen >= 150 (filter out short
        #    free-form items with corrupted answer markers; per Agent 1 finding,
        #    public id=977 is a 106-char trig simplify that falls through).
        #    Per MathArena qwen3_4b_2507.yaml and DeepSeek-R1 README: NO system
        #    message; user-only prompt.
        #    Agent 2 Q3: dropped "multiple valid answers" clause (olympiad
        #    answers are usually unique; offering an escape hatch invites
        #    hedging).
        if n_ans == 0 and qlen >= 150:
            user = (
                "Solve the following problem. Put your final answer within \\boxed{}. "
                "If the answer is a closed-form expression, leave it in exact form "
                "(do not approximate)."
                "\n\n" + question
            )
            user = _append_format_constraints(user, question)
            return None, user, False

        # 4. Long free-form
        if qlen > 500:
            user = _append_format_constraints(question, question)
            return SYSTEM_PROMPT_LONG, user, False

        # 5. Short generic
        user = _append_format_constraints(question, question)
        return SYSTEM_PROMPT_MATH, user, False

    # MCQ branches
    if mcq_needs_options(question, options):
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        user = _append_format_constraints(
            _append_mcq_letter_instruction(f"{question}\n\nOptions:\n{opts_text}"),
            question,
        )
        return SYSTEM_PROMPT_MCQ, user, True

    # MCQ derivation — hide options for stage-1, reuse MATH prompt
    user = _append_format_constraints(question, question)
    return SYSTEM_PROMPT_MATH, user, False


def verify_against_options(response: str, options: list) -> Optional[str]:
    """Strict option matcher (UNUSED in production — kept for diff/A-B comparison).

    Replaced in v2.7+ by verify_against_options_lenient.
    """
    sys.path.insert(0, ".")
    from judger import Judger
    judger_local = Judger(strict_extract=True)
    for i, opt in enumerate(options):
        try:
            if judger_local.auto_judge(pred=response, gold=[opt], options=[[]]):
                return chr(65 + i)
        except Exception:
            continue
    return None


# v2.9 Q8: strip <think>...</think> from stage-1 response when placed in
# stage-2 conversation history. Per Qwen3-Thinking model card: "In multi-turn
# conversations, the historical model output should only include the final
# output part and does not need to include the thinking content."
_STAGE2_THINK_STRIP_RE = re.compile(r"<think>.*?</think>\s*", re.DOTALL)


def build_mcq_stage2_messages(item: dict, stage1_response: str) -> list:
    """Build chat messages for the stage-2 MCQ reconciliation call.

    v2.9 Q8: stage-1 response is stripped of <think> blocks before being
    placed in the assistant history slot (matches Qwen3 chat-template rolling-
    checkpoint behavior and sidesteps the known multi-turn empty-think bug).
    """
    question = item["question"]
    options  = item["options"]
    labels   = [chr(65 + i) for i in range(len(options))]
    opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))

    stage1_stripped = _STAGE2_THINK_STRIP_RE.sub("", stage1_response).strip()

    return [
        {"role": "system",    "content": SYSTEM_PROMPT_MATH},
        {"role": "user",      "content": question},
        {"role": "assistant", "content": stage1_stripped},
        {"role": "user",      "content": f"Options:\n{opts_text}\n\n{MCQ_STAGE2_INSTRUCTION}"},
    ]


def verify_against_options_lenient(
    response: str,
    options: list,
    precision: float = 1e-2,
) -> Optional[str]:
    """Lenient option matcher — does NOT use Judger.auto_judge.

    Picks the BEST match across all options. Strategies tried per option:
      1. Exact cleaned string match → immediate win.
      2. Numeric comparison: relative distance → track minimum.
      3. Sympy SYMBOLIC equality → distance 0.
      4. Decimal-compat at common decimal places → distance 0.

    Internal precision clamped to <=1e-4. MCQ distractors are designed to be
    close (often within 1% of each other); the caller's precision is calibrated
    for free-form scoring and would over-match across multiple options.
    """
    sys.path.insert(0, ".")
    from judger import Judger
    pred = Judger(strict_extract=True).extract_ans(response)
    if not pred:
        return None

    precision = min(precision, 1e-4)

    def _clean(s: str) -> str:
        s = s.strip().strip("$").strip()
        return s.rstrip(";,.!? ").strip()

    _CLEAN_NUM_RE = re.compile(r"^-?(?:\d+(?:\.\d+)?|\.\d+|\d+/\d+)$")
    def _is_clean_number(s: str) -> bool:
        return bool(_CLEAN_NUM_RE.match(s.strip()))

    def _to_float(s: str) -> float:
        s = s.strip().replace(",", "")
        if "/" in s:
            num, den = s.split("/", 1)
            return float(num) / float(den)
        return float(s)

    def _numeric_distance(a: str, b: str) -> Optional[float]:
        if not (_is_clean_number(a) and _is_clean_number(b)):
            return None
        try:
            av, bv = _to_float(a), _to_float(b)
            denom = max(abs(bv), 1e-12)
            return abs(av - bv) / denom
        except (ValueError, TypeError, ZeroDivisionError):
            return None

    def _try_sympy_equality(a: str, b: str) -> bool:
        try:
            from sympy import simplify, sympify
            from sympy.parsing.latex import parse_latex
        except ImportError:
            return False

        def _parse(s: str):
            looks_like_latex = ("\\" in s) or ("{" in s and "}" in s)
            s2 = s.replace("\\cdot", "*").replace("\\times", "*").replace("^", "**")
            if looks_like_latex:
                try:
                    return parse_latex(s)
                except Exception:
                    pass
                try:
                    return sympify(s2)
                except Exception:
                    return None
            else:
                try:
                    return sympify(s2)
                except Exception:
                    pass
                try:
                    return parse_latex(s)
                except Exception:
                    return None

        ae, be = _parse(a), _parse(b)
        if ae is None or be is None:
            return False

        import signal
        class _Timeout(Exception): pass
        def _handler(signum, frame): raise _Timeout()
        old_handler = signal.signal(signal.SIGALRM, _handler)
        signal.alarm(3)
        try:
            return simplify(ae - be) == 0
        except Exception:
            return False
        finally:
            signal.alarm(0)
            signal.signal(signal.SIGALRM, old_handler)

    def _try_decimal_compat(a: str, b: str) -> bool:
        if not (_is_clean_number(a) and _is_clean_number(b)):
            return False
        try:
            av = _to_float(a)
            bv = _to_float(b)
        except (ValueError, TypeError, ZeroDivisionError):
            return False

        def _dp_count(s: str) -> int:
            s_clean = s.strip().replace(",", "").lstrip("-+")
            if "." not in s_clean:
                return 0
            return len(s_clean.split(".")[1].rstrip("0"))

        a_dp, b_dp = _dp_count(a), _dp_count(b)
        common_dp = min(a_dp, b_dp)
        if common_dp == 0:
            return abs(av - bv) < 0.5
        return round(av, common_dp) == round(bv, common_dp)

    pred_norm = _clean(pred)

    best_letter   = None
    best_distance = float("inf")

    for i, opt in enumerate(options):
        letter   = chr(65 + i)
        opt_norm = _clean(opt)

        if pred_norm == opt_norm:
            return letter

        d = _numeric_distance(pred_norm, opt_norm)
        if d is not None:
            if d < precision and d < best_distance:
                best_distance = d
                best_letter   = letter
            continue

        if _try_sympy_equality(pred_norm, opt_norm):
            if 0 < best_distance:
                best_distance = 0
                best_letter   = letter
            continue

        if _try_decimal_compat(pred_norm, opt_norm):
            if 0 < best_distance:
                best_distance = 0
                best_letter   = letter

    return best_letter


_WORD2DIGIT = {
    "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
    "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
}
_NUM_TOKEN = r"(\d+|" + "|".join(_WORD2DIGIT) + r")"


def _parse_num_token(tok: str) -> int:
    return int(tok) if tok.isdigit() else _WORD2DIGIT[tok.lower()]


def infer_precision_from_question(question: str, default: float = 1e-2) -> float:
    """Parse question text for explicit precision spec; return relative tolerance."""
    q = (question or "").lower()
    if re.search(r"\b(?:exact(?:\s+(?:value|answer))?|do not round|no rounding)\b", q):
        return 1e-8
    m = re.search(
        rf"(?:to|correct to|accurate to|round(?:ed)? to|at least|use(?:s|d|ing)?|with)\s+{_NUM_TOKEN}\s+decimal\s+places?",
        q,
    )
    if m:
        return 10 ** (-_parse_num_token(m.group(1)))
    m = re.search(
        rf"(?:to|correct to|accurate to|with|using)\s+{_NUM_TOKEN}\s+(?:significant\s+(?:figures?|digits?)|sig\.?\s*figs?)",
        q,
    )
    if m:
        return 10 ** (-(_parse_num_token(m.group(1)) - 1))
    m = re.search(r"nearest\s+(tenth|hundredth|thousandth|ten-thousandth|hundred-thousandth)", q)
    if m:
        return {"tenth":1e-1,"hundredth":1e-2,"thousandth":1e-3,"ten-thousandth":1e-4,"hundred-thousandth":1e-5}[m.group(1)]
    return default


In [8]:
# Verify on real dataset samples
print("Baseline build_prompt on MCQ sample")
sys_p, usr_p = build_prompt(mcq_sample["question"], mcq_sample.get("options"))
print(f"  system: {sys_p[:60]}...")
print(f"  user  : {usr_p[:100]}... (includes options)")

print("\nselect_prompt on MCQ sample (derivation MCQ — stage 1, no options)")
prepped = preprocess_question(mcq_sample)
sys_p, usr_p, opts_visible = select_prompt(prepped)
print(f"  system: {sys_p[:60]}...")
print(f"  user  : {usr_p[:80]}...")
print(f"  options_visible = {opts_visible}  (False => two-stage flow)")

print("\nRouting on free-form sample (Q0, short)")
prepped = preprocess_question(free_sample)
sys_p, usr_p, opts_visible = select_prompt(prepped)
print(f"  qlen={len(prepped['question'])} n_ans={prepped['question'].count('[ANS]')}")
print(f"  system: {sys_p[:60]}...")
print(f"  options_visible = {opts_visible}")

print("\nHeuristic check on an options-required MCQ (id 281, 'which of the following ... is unbiased')")
opt_required_sample = next((d for d in data if d.get("id") == 281), None)
if opt_required_sample is not None:
    prepped = preprocess_question(opt_required_sample)
    sys_p, usr_p, opts_visible = select_prompt(prepped)
    print(f"  system: {sys_p[:60]}...")
    print(f"  user  : {usr_p[:120]}...")
    print(f"  options_visible = {opts_visible}  (True => single-stage with options inline)")
else:
    print("  (id 281 not present in this dataset)")

print("\nStage-2 message structure (mock — no real stage-1 response yet)")
mock_stage1 = "After computing the integral, I get \\boxed{2/a}."
msgs = build_mcq_stage2_messages(preprocess_question(mcq_sample), mock_stage1)
for m in msgs:
    print(f"  [{m['role']:9s}] {m['content'][:80]}...")


Baseline build_prompt on MCQ sample
  system: Please reason step by step, and put your final answer within...
  user  : $int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}... (includes options)

select_prompt on MCQ sample (derivation MCQ — stage 1, no options)
  system: Please reason step by step, and put your final answer within...
  user  : $\int_{-\infty}^{+\infty} \frac{a^{3/2}}{s^2+a^2} ds = $...
  options_visible = False  (False => two-stage flow)

Routing on free-form sample (Q0, short)
  qlen=71 n_ans=1
  system: Please reason step by step, and put your final answer within...
  options_visible = False

Heuristic check on an options-required MCQ (id 281, 'which of the following ... is unbiased')
  system: Please reason step by step, and put your final answer within...
  user  : Let the population (X) be uniformly distributed on ([0, \theta]). A sample of size 1, (X_1), is drawn from this populati...
  options_visible = True  (Tru

## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# v2.9: max_model_len bumped 49152 → 90112 to hold MAX_TOKENS_OLYMPIAD=81920
# output + olympiad input headroom. max_num_seqs halved 64 → 32 because 24 GB
# A30 cannot fit 64 sequences at 90K context; vLLM PagedAttention handles
# dynamic allocation but the scheduler still respects max_num_seqs upper bound.
llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.90,
    max_model_len=90112,
    trust_remote_code=True,
    max_num_seqs=32,
    max_num_batched_tokens=16384,
    enforce_eager=False,
)

# Default sampling — used for everything EXCEPT olympiad branch
sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    seed=SEED,
)

# v2.9: olympiad-only sampling. MAX_TOKENS=81920 per MathArena's qwen3_4b_2507.yaml
# and Qwen3-4B-Thinking-2507 model card recommendation for "math and programming
# competitions" (vs 32768 for normal queries). Used only when select_prompt
# returns sys_p=None (olympiad branch).
MAX_TOKENS_OLYMPIAD = 81920
sampling_params_olympiad = SamplingParams(
    max_tokens=MAX_TOKENS_OLYMPIAD,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    seed=SEED,
)

print("Model loaded.")


INFO 05-29 22:19:40 [utils.py:233] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 90112, 'enable_prefix_caching': False, 'max_num_batched_tokens': 16384, 'max_num_seqs': 32, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'Qwen/Qwen3-4B-Thinking-2507'}


INFO 05-29 22:19:54 [model.py:549] Resolved architecture: Qwen3ForCausalLM


INFO 05-29 22:19:54 [model.py:1678] Using max model len 90112


INFO 05-29 22:19:54 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=16384.


INFO 05-29 22:19:54 [vllm.py:790] Asynchronous scheduling is enabled.


...

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 11/11 [00:01<00:00, 10.10it/s]
Capturing CUDA graphs (decode, FULL):  14%|█▍        | 1/7 [00:00<00:00,  9.84it/s]

Capturing CUDA graphs (decode, FULL):  71%|███████▏  | 5/7 [00:00<00:00, 10.20it/s]

Capturing CUDA graphs (decode, FULL): 100%|██████████| 7/7 [00:00<00:00, 10.74it/s]


(EngineCore pid=329) INFO 05-29 22:20:54 [gpu_model_runner.py:6046] Graph capturing finished in 3 secs, took 1.04 GiB
(EngineCore pid=329) INFO 05-29 22:20:54 [gpu_worker.py:597] CUDA graph pool memory: 1.04 GiB (actual), 1.33 GiB (estimated), difference: 0.29 GiB (28.1%).
(EngineCore pid=329) INFO 05-29 22:20:54 [core.py:283] init engine (profile, create kv cache, warmup model) took 34.13 seconds


(EngineCore pid=329) INFO 05-29 22:20:55 [vllm.py:790] Asynchronous scheduling is enabled.
Model loaded.


In [10]:
# # from new starter code --- will be useful once we enter SFT phase
# import torch
# from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [11]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Generate
# print(f"Generating responses for {len(prompts)} questions...")
# outputs = llm.generate(prompts, sampling_params=sampling_params)

# responses = [out.outputs[0].text.strip() for out in outputs]

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")


In [12]:
import hashlib

# Build/Load the stratified eval subset ids
if Path(SUBSET_PATH).exists():
    with open(SUBSET_PATH) as f:
        subset_ids = json.load(f)
    print(f"Loaded existing subset of {len(subset_ids)} IDs from {SUBSET_PATH}")
else:
    import random
    rng = random.Random(SEED)

    def stratify_by_length(items, n):
        items = sorted(items, key=lambda d: len(d["question"]))
        b = len(items) // 3
        buckets = [items[:b], items[b:2*b], items[2*b:]] # split into 3 pools, each represent the specifc length portion
        base, rem = divmod(n, 3) # pick base number of questions from each pool
        chosen = []
        for i, bucket in enumerate(buckets):
            k = base + (1 if i < rem else 0)
            chosen.extend(rng.sample(bucket, k))
        return chosen

    mcq_pool  = [d for d in data if d.get("options")]
    free_pool = [d for d in data if not d.get("options")]
    chosen = stratify_by_length(mcq_pool, EVAL_N_MCQ) + stratify_by_length(free_pool, EVAL_N_FREE)
    subset_ids = [d["id"] for d in chosen]

    Path(SUBSET_PATH).parent.mkdir(parents=True, exist_ok=True)
    with open(SUBSET_PATH, "w") as f:
        json.dump(subset_ids, f, indent=2)
    print(f"Saved new subset of {len(subset_ids)} IDs to {SUBSET_PATH}")

Loaded existing subset of 100 IDs from results/eval_subset.json


In [13]:
# Load eval subset with ids
id_set = set(subset_ids)
subset = [d for d in data if d["id"] in id_set]
n_mcq  = sum(1 for d in subset if d.get("options"))
print(f"Eval subset: {len(subset)} questions ({n_mcq} MCQ, {len(subset)-n_mcq} free-form)")

Eval subset: 100 questions (50 MCQ, 50 free-form)


For save/resume from interuption

In [14]:
# Cache key: include all v2 prompts + pipeline tag so prior caches survive
PIPELINE_TAG = "v2.9"
prompt_hash = hashlib.md5(
    (SYSTEM_PROMPT_MATH + "||" + SYSTEM_PROMPT_MCQ + "||"
     + MCQ_STAGE2_INSTRUCTION + "||" + SYSTEM_PROMPT_LONG + "||"
     + SYSTEM_PROMPT_MULTIPART + "||" + PIPELINE_TAG).encode()
).hexdigest()[:8]
CACHE_PATH = f"results/cache/{prompt_hash}_seed{SEED}.jsonl"
Path(CACHE_PATH).parent.mkdir(parents=True, exist_ok=True)

# Cache loader: only read 'response' (scorer needs); ignore 'stage1' (diagnostic)
cache = {}
if Path(CACHE_PATH).exists():
    with open(CACHE_PATH) as f:
        for line in f:
            e = json.loads(line)
            cache[e["id"]] = e["response"]
print(f"Cache hits: {len(cache)} for prompt hash {prompt_hash} ({PIPELINE_TAG})")

to_generate = [item for item in subset if item["id"] not in cache]
n_to_gen_mcq  = sum(1 for it in to_generate if it.get("options"))
print(f"Need to generate: {len(to_generate)} ({n_to_gen_mcq} MCQ stage-1, {len(to_generate)-n_to_gen_mcq} free-form)")


Cache hits: 32 for prompt hash f6986b4d (v2.9)
Need to generate: 68 (29 MCQ stage-1, 39 free-form)


In [15]:
# v2.9 mini-batched generation with heuristic-routed MCQ flow (crash-safe)
# Per batch:
# 1. Stage-1: generate for ALL items (handles None system prompt for olympiad)
# 2. MathArena last-chance reprompt for any stage-1 response missing \boxed{}
#    after </think> (with MCQ-options appendage for visible-MCQ items, Q7)
# 3. Classify: free-form / MCQ visible-opts / MCQ-hidden s1-hit / needs stage-2
# 4. Stage-2: generate ONLY for hidden-options MCQs where verify failed
# 5. Append batch results to cache file (crash-safe)
BATCH_SIZE = 16

# v2.9: MathArena last-chance reprompt (verbatim base from
# https://github.com/eth-sri/matharena/blob/main/src/matharena/runner.py)
LAST_CHANCE_REPROMPT_BASE = (
    "Your last message does not provide a final answer in a way that follows the "
    "formatting instructions. Please based on the conversation history, report the "
    "final answer again within \\boxed{}. If you did not find the answer, please use "
    "\\boxed{None}. Do not reason about the problem again or use tools, simply try "
    "to extract the final answer from the previous reasoning. Boxed answers in "
    "thinking/reasoning stages will be ignored; only the final response message is "
    "considered."
)

# Q7: MCQ-options appendage — fired for visible-MCQ items only (where the
# model has the option set in context). For hidden-options MCQ stage-1, the
# model is supposed to derive an open answer; appending options would bias it.
def _build_reprompt(options_visible: bool, options_list: Optional[list]) -> str:
    if options_visible and options_list:
        labels   = [chr(65 + i) for i in range(len(options_list))]
        opts_str = ", ".join(f"\\boxed{{{l}}}" for l in labels)
        return (
            LAST_CHANCE_REPROMPT_BASE
            + f" Recall that this is a multiple choice problem, and the only valid "
              f"answers to put within \\boxed{{}} are: {opts_str}."
        )
    return LAST_CHANCE_REPROMPT_BASE


# Reprompt sampling: low temp + small budget, single-shot extraction
sampling_params_reprompt = SamplingParams(
    max_tokens=2048,
    temperature=0.0,
    top_p=1.0,
    seed=SEED,
    n=1,
)

_THINK_BLOCK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def _build_chat(sys_p, user_p):
    """Build messages list, omitting system role when sys_p is None (olympiad)."""
    if sys_p is None:
        return [{"role": "user", "content": user_p}]
    return [
        {"role": "system", "content": sys_p},
        {"role": "user",   "content": user_p},
    ]


def _has_final_box(response: str) -> bool:
    """True iff a \\boxed{...} exists OUTSIDE <think>...</think>."""
    text = _THINK_BLOCK_RE.sub("", response)
    return bool(re.search(r"\\boxed\{", text))


total_batches = (len(to_generate) + BATCH_SIZE - 1) // BATCH_SIZE if to_generate else 0
total_mcq_visible_opts = 0
total_mcq_s1_hits = 0
total_stage2_calls = 0
total_reprompts = 0

for batch_start in range(0, len(to_generate), BATCH_SIZE):
    batch = to_generate[batch_start:batch_start + BATCH_SIZE]
    batch_num = batch_start // BATCH_SIZE + 1
    prepped = [preprocess_question(item) for item in batch]

    # Stage 1
    stage1_prompts = []
    options_visible_flags = []
    sys_user_pairs = []   # cached for reprompt path
    for p in prepped:
        sys_p, user_p, opts_visible = select_prompt(p)
        options_visible_flags.append(opts_visible)
        sys_user_pairs.append((sys_p, user_p))
        stage1_prompts.append(tokenizer.apply_chat_template(
            _build_chat(sys_p, user_p),
            tokenize=False,
            add_generation_prompt=True,
        ))
    # v2.9: per-prompt sampling — olympiad branch gets MAX_TOKENS=81920
    # (MathArena recommendation for Qwen3-4B-Thinking on competition problems);
    # all other branches use the default sampling_params (MAX_TOKENS=32768).
    sp_list = [sampling_params_olympiad if sys_p is None else sampling_params
               for sys_p, _ in sys_user_pairs]
    stage1_outs = llm.generate(stage1_prompts, sampling_params=sp_list)
    stage1_texts = [o.outputs[0].text.strip() for o in stage1_outs]

    # v2.9 §P.8: MathArena last-chance reprompt for items with no \boxed{} after </think>
    reprompt_idx = [i for i, s1 in enumerate(stage1_texts) if not _has_final_box(s1)]
    if reprompt_idx:
        reprompt_prompts = []
        for i in reprompt_idx:
            sys_p, user_p = sys_user_pairs[i]
            # Visible-MCQ items get the MCQ-options appendage (Q7)
            opts_list = prepped[i].get("options") if options_visible_flags[i] else None
            # For embedded-MCQ (no item['options'] but options_visible=True),
            # detect_embedded_mcq gives us the inline options
            if options_visible_flags[i] and not opts_list:
                opts_list = detect_embedded_mcq(prepped[i]["question"])
            reprompt_user = _build_reprompt(options_visible_flags[i], opts_list)
            base_msgs = _build_chat(sys_p, user_p)
            msgs = base_msgs + [
                {"role": "assistant", "content": stage1_texts[i]},
                {"role": "user",      "content": reprompt_user},
            ]
            reprompt_prompts.append(tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True,
            ))
        reprompt_outs = llm.generate(reprompt_prompts, sampling_params=sampling_params_reprompt)
        for k, i in enumerate(reprompt_idx):
            # Append reprompt's extracted answer so judger's "last contiguous
            # \boxed{} group" picks it up.
            stage1_texts[i] = stage1_texts[i] + "\n\n" + reprompt_outs[k].outputs[0].text.strip()

    # Classify
    final_response = [None] * len(batch)
    stage1_field   = [None] * len(batch)
    needs_stage2_idx = []
    n_mcq_visible_opts = 0
    n_mcq_s1_hits = 0
    n_freeform = 0

    for i, (orig_item, prep_item, s1) in enumerate(zip(batch, prepped, stage1_texts)):
        if not orig_item.get("options") and not options_visible_flags[i]:
            # Free-form (incl. olympiad) and zero-option items: stage-1 IS the answer
            final_response[i] = s1
            n_freeform += 1
        elif options_visible_flags[i]:
            # MCQ visible-opts (incl. embedded-MCQ branch): trust the model's letter
            final_response[i] = s1
            n_mcq_visible_opts += 1
        else:
            # MCQ derivation (hidden-options): match stage-1 against options
            stage1_field[i] = s1
            item_precision = infer_precision_from_question(
                prep_item["question"], default=EVAL_PRECISION
            )
            letter = verify_against_options_lenient(
                s1, prep_item["options"], precision=item_precision
            )
            if letter is not None:
                final_response[i] = f"\\boxed{{{letter}}}"
                n_mcq_s1_hits += 1
            else:
                needs_stage2_idx.append(i)

    # Stage 2: only call llm.generate if there is real work
    if needs_stage2_idx:
        s2_prompts = []
        for i in needs_stage2_idx:
            msgs = build_mcq_stage2_messages(prepped[i], stage1_texts[i])
            s2_prompts.append(tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True,
            ))
        s2_outs = llm.generate(s2_prompts, sampling_params=sampling_params)
        for k, i in enumerate(needs_stage2_idx):
            final_response[i] = s2_outs[k].outputs[0].text.strip()

    # Append-save the batch (crash-safe)
    with open(CACHE_PATH, "a") as f:
        for i, item in enumerate(batch):
            rec = {"id": item["id"], "response": final_response[i]}
            if stage1_field[i] is not None:
                rec["stage1"] = stage1_field[i]
            cache[item["id"]] = final_response[i]
            f.write(json.dumps(rec) + "\n")

    total_mcq_visible_opts += n_mcq_visible_opts
    total_mcq_s1_hits += n_mcq_s1_hits
    total_stage2_calls += len(needs_stage2_idx)
    total_reprompts += len(reprompt_idx)
    print(f"Batch {batch_num}/{total_batches}: "
          f"MCQ visible-opts={n_mcq_visible_opts}, hidden s1-hit={n_mcq_s1_hits}, "
          f"stage-2={len(needs_stage2_idx)}, free-form={n_freeform}, "
          f"reprompts={len(reprompt_idx)} | total cached={len(cache)}/{len(subset)}")

if to_generate:
    print(f"\nGeneration complete. MCQ visible-opts: {total_mcq_visible_opts}, "
          f"hidden s1-hits: {total_mcq_s1_hits}, stage-2 calls: {total_stage2_calls}, "
          f"last-chance reprompts: {total_reprompts}")

# Align responses with subset order
responses = [cache[item["id"]] for item in subset]
print(f"Ready: {len(responses)} responses for scoring")

# Preview first 2 responses
for i in range(min(2, len(responses))):
    print(f"\n── Response (id={subset[i]['id']}, type={'MCQ' if subset[i].get('options') else 'free-form'}) ──")
    print(responses[i][:300], "..." if len(responses[i]) > 300 else "")


Rendering prompts:   0%|          | 0/16 [00:00<?, ?it/s]

Rendering prompts: 100%|██████████| 16/16 [00:00<00:00, 395.37it/s]

Processed prompts:   0%|          | 0/16 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   6%|▋         | 1/16 [00:12<03:05, 12.34s/it, est. speed input: 17.83 toks/s, output: 26.90 toks/s]

Processed prompts:  12%|█▎        | 2/16 [00:29<03:35, 15.41s/it, est. speed input: 10.10 toks/s, output: 33.45 toks/s]

...

Processed prompts:  75%|███████▌  | 3/4 [18:19<07:25, 445.67s/it, est. speed input: 1.90 toks/s, output: 38.63 toks/s]

Processed prompts: 100%|██████████| 4/4 [19:21<00:00, 293.97s/it, est. speed input: 1.98 toks/s, output: 64.55 toks/s]

Processed prompts: 100%|██████████| 4/4 [19:21<00:00, 293.97s/it, est. speed input: 1.98 toks/s, output: 64.55 toks/s]

Processed prompts: 100%|██████████| 4/4 [19:21<00:00, 290.26s/it, est. speed input: 1.98 toks/s, output: 64.55 toks/s]

Batch 5/5: MCQ visible-opts=1, hidden s1-hit=0, stage-2=0, free-form=3, reprompts=0 | total cached=100/100

Generation complete. MCQ visible-opts: 10, hidden s1-hits: 11, stage-2 calls: 13, last-chance reprompts: 1
Ready: 100 responses for scoring

── Response (id=1, type=MCQ) ──
Okay, let's go through this again carefully. The original integral is ∫ from -∞ to +∞ of (a^(3/2))/(s² + a²) ds. 

First, I think the user might have made a mistake in the options. Let me check the calculation again.

The integral of 1/(s² + a²) from -∞ to ∞ is π/a, right? So multiplying by a^(3/2)  ...

── Response (id=4, type=MCQ) ──
Okay, let's go through this again carefully. The problem says to find the analytic function f(z) = u + iv where u is given as x³ + 6x²y - 3xy² - 2y³, and f(0) = 0.

First, I need to check if my previous answer was correct. I thought it was (1 - 2i)z³, which is option C. But the user says that the pr ...


In [16]:
# Smoke test: lightweight debug checks before scoring
# Catches obvious problems (length mismatch, missing boxed letters) without
# running the full Judger pipeline.

# 1. responses aligned with subset
assert len(responses) == len(subset), \
    f"Length mismatch: {len(responses)} responses vs {len(subset)} subset items"

# 2. every MCQ has a single-letter \boxed{} somewhere in its response
mcq_missing_box = []
for item, resp in zip(subset, responses):
    if item.get("options") and not re.search(r"\\boxed\{[A-Z]\}", resp):
        mcq_missing_box.append(item["id"])
if mcq_missing_box:
    print(f"WARNING: {len(mcq_missing_box)} MCQ items have no single-letter \\boxed: {mcq_missing_box}")
    print("  (scorer will fall back to last standalone capital letter — acceptable but worth noting)")
else:
    print(f"OK: all {sum(1 for d in subset if d.get('options'))} MCQ items have a \\boxed{{LETTER}}")

# 3. Show which MCQ ids took the visible-options route (heuristic-flagged)
visible_opt_ids = [
    item["id"] for item in subset
    if item.get("options") and mcq_needs_options(item["question"], item["options"])
]
print(f"\nMCQ items routed visible-options (n={len(visible_opt_ids)}): {visible_opt_ids}")

# 4. Eyeball one stage-1-hit and one stage-2-used MCQ entry from the cache
print("\nSample cache entries (for human inspection):")
sample_count = 0
with open(CACHE_PATH) as f:
    for line in f:
        if sample_count >= 2:
            break
        e = json.loads(line)
        if "stage1" not in e:
            continue
        is_synthetic = e["response"].startswith("\\boxed{") and len(e["response"]) <= 12
        kind = "stage-1 hit (synthetic \\boxed{})" if is_synthetic else "stage-2 used (full response)"
        resp_preview   = e["response"][:120] + ("..." if len(e["response"]) > 120 else "")
        stage1_preview = e["stage1"][:120]   + ("..." if len(e["stage1"])   > 120 else "")
        print(f"\n  [id={e['id']}] {kind}")
        print(f"    response: {resp_preview}")
        print(f"    stage1  : {stage1_preview}")
        sample_count += 1


OK: all 50 MCQ items have a \boxed{LETTER}

MCQ items routed visible-options (n=10): [154, 182, 232, 233, 235, 480, 800, 857, 959, 1110]

Sample cache entries (for human inspection):

  [id=1] stage-2 used (full response)
    response: Okay, let's go through this again carefully. The original integral is ∫ from -∞ to +∞ of (a^(3/2))/(s² + a²) ds. 

First...
    stage1  : Okay, let's try to figure out how to solve this integral: the integral from negative infinity to positive infinity of (a...

  [id=4] stage-2 used (full response)
    response: Okay, let's go through this again carefully. The problem says to find the analytic function f(z) = u + iv where u is giv...
    stage1  : Okay, let's see. I need to find the analytic function \( f(z) = u + iv \) where \( u(x, y) = x^3 + 6x^2y - 3xy^2 - 2y^3 ...


## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [17]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(subset, responses), total=len(subset), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        # Per-question precision (e.g. "to 3 decimal places" → 1e-3, "exact" → 1e-8);
        # falls back to EVAL_PRECISION when the question doesn't specify.
        item_precision = infer_precision_from_question(
            item["question"], default=EVAL_PRECISION
        )
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
                precision=item_precision,
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring:   0%|          | 0/100 [00:00<?, ?it/s]

Scoring:  25%|██▌       | 25/100 [00:00<00:00, 158.12it/s]

Scoring:  41%|████      | 41/100 [00:00<00:00, 120.40it/s]

Scoring:  58%|█████▊    | 58/100 [00:00<00:00, 124.47it/s]

Scoring:  71%|███████   | 71/100 [00:00<00:00, 106.61it/s]

Scoring:  82%|████████▏ | 82/100 [00:01<00:00, 43.62it/s] 

Scoring:  91%|█████████ | 91/100 [00:01<00:00, 42.33it/s]

Scoring:  98%|█████████▊| 98/100 [00:01<00:00, 44.50it/s]

Scoring: 100%|██████████| 100/100 [00:01<00:00, 53.05it/s]

Scoring complete. 100 results.


## 8. Summary

Print accuracy broken down by question type.

In [18]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

# Per-length-bucket accuracy (within stratified subset)
sorted_q = sorted(subset, key=lambda d: len(d["question"]))
bsize = len(sorted_q) // 3
length_buckets = {
    "short ": [d["id"] for d in sorted_q[:bsize]],
    "medium": [d["id"] for d in sorted_q[bsize:2*bsize]],
    "long  ": [d["id"] for d in sorted_q[2*bsize:]],
}
print("\nBy question length:")
for label, ids in length_buckets.items():
    s = [r for r in results if r["id"] in set(ids)]
    if s:
        print(f"  {label}: {sum(r['correct'] for r in s):2d} / {len(s):2d}  ({acc(s):.2f}%)")

# Per-routing-path accuracy (MCQ only) — diagnoses whether the heuristic
# is paying off vs the two-stage hidden-options flow.
print("\nBy routing path (MCQ only):")
for label, pred in [
    ("visible-opts", lambda it: mcq_needs_options(it["question"], it["options"])),
    ("hidden-opts ", lambda it: not mcq_needs_options(it["question"], it["options"])),
]:
    ids = {it["id"] for it in subset if it.get("options") and pred(it)}
    s = [r for r in results if r["id"] in ids]
    if s:
        print(f"  {label}: {sum(r['correct'] for r in s):2d} / {len(s):2d}  ({acc(s):.2f}%)")


EVALUATION RESULTS
  MCQ        :   33 /   50  (66.00%)
  Free-form  :   29 /   50  (58.00%)
  Overall    :   62 /  100  (62.00%)

By question length:
  short : 24 / 33  (72.73%)
  medium: 19 / 33  (57.58%)
  long  : 19 / 34  (55.88%)

By routing path (MCQ only):
  visible-opts:  6 / 10  (60.00%)
  hidden-opts : 27 / 40  (67.50%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [19]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 100 records to results/starter_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!